# Category 3 — Binary Search on Answer (Optimization)

---

## The Paradigm Shift

> You are no longer searching an array. You are searching the **space of possible answers**.

The template is always:

$$\text{define } [lo, hi] \text{ on the answer} \quad \rightarrow \quad \text{write monotonic } \texttt{check(mid)} \quad \rightarrow \quad \text{BS for the boundary}$$

`check(mid)` is always a **yes/no feasibility predicate**. The answer space always looks like:

```
Minimize:  F F F F [T T T T T]   →  first True
Maximize:  [T T T T T] F F F F   →  last True
```

---

## Group 1 — Direct Monotonic Functions

**Square Root (integer):** Find largest $x$ where $x^2 \leq n$.

$$lo=1,\ hi=n \qquad \text{check}(mid): mid^2 \leq n \qquad \text{→ last True (maximize)}$$

**Precision sqrt:** Once integer part found, do 6 iterations of decimal refinement with step $0.1, 0.01 \ldots$

**Nth Root:** Find largest $x$ where $x^n \leq m$. Same skeleton, `check` uses `pow(mid, n)`.

> **Overflow delta:** Never compute $mid^n$ directly. Use a multiplicative loop with early exit — if intermediate product exceeds $m$, return false immediately.

---

## Group 2 — Resource Allocation (Minimize the constraint)

These all share one skeleton. The predicate is: *"can we satisfy the constraint with this `mid` value?"*

| Problem | $lo$ | $hi$ | `check(mid)` predicate | Direction |
|---|---|---|---|---|
| Koko Eating Bananas | $1$ | $\max(piles)$ | $\sum \lceil pile_i / mid \rceil \leq h$ | minimize |
| Min Days for M Bouquets | $1$ | $\max(bloom)$ | count consecutive bloomed flowers $\geq m \times k$ | minimize |
| Smallest Divisor | $1$ | $\max(arr)$ | $\sum \lceil arr_i / mid \rceil \leq threshold$ | minimize |
| Ship Packages in D Days | $\max(wt)$ | $\sum wt$ | greedily pack, days used $\leq D$ | minimize |

**The delta — why `lo = max(arr)` for ship packages?**
You can never ship if capacity $<$ heaviest package. It's a hard lower bound, not $1$.

---

## Group 3 — Partition Problems (Same predicate, different framing)

**Book Allocation / Painters Partition / Split Array Largest Sum** — these are **identical problems** with different nouns.

$$\text{minimize the maximum sum of any partition into } k \text{ contiguous subarrays}$$

$$lo = \max(arr), \quad hi = \sum arr$$

`check(mid)`: greedily assign elements; start new partition when adding next element exceeds `mid`. Count partitions $\leq k$.

**Aggressive Cows** — the **mirror**: maximize the minimum distance.

$$lo = 1, \quad hi = arr[last] - arr[0] \quad \text{(after sorting)}$$

`check(mid)`: greedily place cows; place next cow only when gap $\geq mid$. Count placed cows $\geq k$.

> **Mirror delta:** Partition problems minimize-the-max → first True. Aggressive Cows maximizes-the-min → last True. The `ans` update flips: `ans = mid; lo = mid + 1` instead of `ans = mid; hi = mid - 1`.

---

## Group 4 — Minimize Max Distance to Gas Station

**The only real-valued BS in this category.**

$$lo = 0.0, \quad hi = \max(\text{gaps between stations})$$

`check(mid)`: for each gap $g$, you need $\lfloor g/mid \rfloor$ stations inserted. Total inserted $\leq k$.

**The delta:** Loop runs a fixed number of iterations (e.g., 100) instead of `lo < hi`. Termination is by precision, not by convergence.

---

## Group 5 — Median of Two Sorted Arrays $O(\log \min(n,m))$

**Core idea:** Partition both arrays such that the left half of the merged array has $\frac{n+m+1}{2}$ elements total.

$$\text{Binary search on partition of smaller array: } cut_1 \in [0, n]$$
$$cut_2 = \frac{n+m+1}{2} - cut_1$$

**Valid partition condition:**
$$l_1 \leq r_2 \quad \text{and} \quad l_2 \leq r_1$$

where $l_1 = A[cut_1-1],\ r_1 = A[cut_1],\ l_2 = B[cut_2-1],\ r_2 = B[cut_2]$.

**Boundary transitions:**

$$l_1 > r_2 \Rightarrow cut_1 \text{ too far right} \Rightarrow hi = cut_1 - 1$$
$$l_2 > r_1 \Rightarrow cut_1 \text{ too far left} \Rightarrow lo = cut_1 + 1$$

**Answer:**
$$\text{odd total} \Rightarrow \max(l_1, l_2) \qquad \text{even total} \Rightarrow \frac{\max(l_1,l_2) + \min(r_1,r_2)}{2}$$

**The delta — why always BS on the smaller array?**
$cut_2$ is derived from $cut_1$. If $n > m$, $cut_2$ can go negative → invalid. Always ensure $n \leq m$ by swapping.

---

## The Master Template (Internalize This)

```
lo = minimum possible answer  (hard constraint, not 0 or 1 blindly)
hi = maximum possible answer  (sum, max, or range)

while lo <= hi:
    mid = lo + (hi - lo) / 2
    if check(mid):
        ans = mid
        hi = mid - 1   ← minimize (first True)
        OR
        lo = mid + 1   ← maximize (last True)
    else:
        lo = mid + 1   ← minimize
        OR
        hi = mid - 1   ← maximize
```

In [1]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <numeric>
#include <iomanip>

using namespace std;

In [2]:
// ─── Square Root (Integer + Precision) ───────────────────────────────────
long long sqrtInt(long long n) {
    long long lo = 1, hi = n, ans = 1;
    while (lo <= hi) {
        long long mid = lo + (hi - lo) / 2;
        if (mid * mid <= n) { ans = mid; lo = mid + 1; }
        else                  hi = mid - 1;
    }
    return ans;
}

double sqrtPrecision(int n, int precision = 6) {
    double lo = 0, hi = n;
    double step = 1.0;
    for (int p = 0; p < precision; p++) {
        while (lo + step <= hi && (lo + step) * (lo + step) <= n)
            lo += step;
        step /= 10.0;
    }
    return lo;
}

// ─── Nth Root (overflow-safe check) ───────────────────────────────────────
bool nthRootCheck(long long mid, int n, long long m) {
    long long val = 1;
    for (int i = 0; i < n; i++) {
        val *= mid;
        if (val > m) return false; // early exit — overflow guard
    }
    return val == m;
}

int nthRoot(int n, long long m) {
    long long lo = 1, hi = m;
    while (lo <= hi) {
        long long mid = lo + (hi - lo) / 2;
        if (nthRootCheck(mid, n, m))  return mid;
        // compute mid^n > m or < m
        long long val = 1; bool over = false;
        for (int i = 0; i < n; i++) { val *= mid; if (val > m) { over = true; break; } }
        if (over) hi = mid - 1;
        else      lo = mid + 1;
    }
    return -1;
}

// ─── Koko Eating Bananas ──────────────────────────────────────────────────
bool kokoCheck(vector<int>& piles, int speed, int h) {
    long long hours = 0;
    for (int p : piles) hours += (p + speed - 1) / speed; // ceil(p/speed)
    return hours <= h;
}

int minEatingSpeed(vector<int>& piles, int h) {
    int lo = 1, hi = *max_element(piles.begin(), piles.end()), ans = hi;
    while (lo <= hi) {
        int mid = lo + (hi - lo) / 2;
        if (kokoCheck(piles, mid, h)) { ans = mid; hi = mid - 1; }
        else                            lo = mid + 1;
    }
    return ans;
}

// ─── Ship Packages in D Days ──────────────────────────────────────────────
// lo = max(weights) — hard lower bound
bool shipCheck(vector<int>& wt, int cap, int days) {
    int d = 1, cur = 0;
    for (int w : wt) {
        if (cur + w > cap) { d++; cur = 0; }
        cur += w;
    }
    return d <= days;
}

int shipCapacity(vector<int>& wt, int days) {
    int lo = *max_element(wt.begin(), wt.end());
    int hi = accumulate(wt.begin(), wt.end(), 0), ans = hi;
    while (lo <= hi) {
        int mid = lo + (hi - lo) / 2;
        if (shipCheck(wt, mid, days)) { ans = mid; hi = mid - 1; }
        else                            lo = mid + 1;
    }
    return ans;
}

// ─── Book Allocation / Split Array Largest Sum ────────────────────────────
// Identical predicate for painters partition too — just rename variables
bool allocCheck(vector<int>& arr, int maxSum, int k) {
    int partitions = 1, cur = 0;
    for (int x : arr) {
        if (cur + x > maxSum) { partitions++; cur = 0; }
        cur += x;
    }
    return partitions <= k;
}

int bookAllocation(vector<int>& pages, int k) {
    if (k > (int)pages.size()) return -1;
    int lo = *max_element(pages.begin(), pages.end());
    int hi = accumulate(pages.begin(), pages.end(), 0), ans = hi;
    while (lo <= hi) {
        int mid = lo + (hi - lo) / 2;
        if (allocCheck(pages, mid, k)) { ans = mid; hi = mid - 1; }
        else                             lo = mid + 1;
    }
    return ans;
}

// ─── Aggressive Cows (maximize minimum) ──────────────────────────────────
// Mirror of partition — last True instead of first True
bool cowsCheck(vector<int>& stalls, int dist, int cows) {
    int placed = 1, last = stalls[0];
    for (int i = 1; i < (int)stalls.size(); i++) {
        if (stalls[i] - last >= dist) { placed++; last = stalls[i]; }
    }
    return placed >= cows;
}

int aggressiveCows(vector<int>& stalls, int k) {
    sort(stalls.begin(), stalls.end());
    int lo = 1, hi = stalls.back() - stalls[0], ans = 1;
    while (lo <= hi) {
        int mid = lo + (hi - lo) / 2;
        if (cowsCheck(stalls, mid, k)) { ans = mid; lo = mid + 1; } // maximize → go right
        else                             hi = mid - 1;
    }
    return ans;
}

// ─── Minimize Max Distance to Gas Station ────────────────────────────────
// Real-valued BS: fixed iterations for precision
double minMaxGasDist(vector<int>& stations, int k) {
    double lo = 0, hi = 0;
    for (int i = 1; i < (int)stations.size(); i++)
        hi = max(hi, (double)(stations[i] - stations[i-1]));

    for (int iter = 0; iter < 100; iter++) {
        double mid = (lo + hi) / 2.0;
        int placed = 0;
        for (int i = 1; i < (int)stations.size(); i++)
            placed += (int)((stations[i] - stations[i-1]) / mid); // floor = sections-1
        if (placed > k) lo = mid;
        else            hi = mid;
    }
    return hi;
}

// ─── Median of Two Sorted Arrays O(log min(n,m)) ─────────────────────────
double findMedianSortedArrays(vector<int>& A, vector<int>& B) {
    if (A.size() > B.size()) return findMedianSortedArrays(B, A); // always BS on smaller
    int n = A.size(), m = B.size();
    int lo = 0, hi = n;
    int half = (n + m + 1) / 2;

    while (lo <= hi) {
        int cut1 = lo + (hi - lo) / 2;
        int cut2 = half - cut1;

        int l1 = (cut1 == 0) ? INT_MIN : A[cut1 - 1];
        int r1 = (cut1 == n) ? INT_MAX : A[cut1];
        int l2 = (cut2 == 0) ? INT_MIN : B[cut2 - 1];
        int r2 = (cut2 == m) ? INT_MAX : B[cut2];

        if      (l1 > r2) hi = cut1 - 1;       // cut1 too far right
        else if (l2 > r1) lo = cut1 + 1;       // cut1 too far left
        else {                                   // valid partition
            if ((n + m) % 2 == 1) return max(l1, l2);
            return (max(l1, l2) + min(r1, r2)) / 2.0;
        }
    }
    return 0.0;
}

In [3]:
    cout << sqrtInt(37)                              << "\n"; // 6
    cout << fixed << setprecision(6) << sqrtPrecision(37) << "\n"; // 6.082762

    vector<int> piles = {3,6,7,11}; cout << minEatingSpeed(piles, 8) << "\n"; // 4
    vector<int> wt    = {1,2,3,4,5,6,7,8,9,10}; cout << shipCapacity(wt, 5) << "\n"; // 15
    vector<int> pages = {12,34,67,90}; cout << bookAllocation(pages, 2) << "\n"; // 113
    vector<int> stalls= {0,3,4,7,10,9}; cout << aggressiveCows(stalls, 4) << "\n"; // 3

    vector<int> a = {1,3}, b = {2};
    cout << findMedianSortedArrays(a, b) << "\n"; // 2.0
    vector<int> c = {1,2}, d = {3,4};
    cout << findMedianSortedArrays(c, d) << "\n"; // 2.5

6
6.082760
4
15
113
3
2.000000
2.500000
